# 02 — Akuisisi data kota: API dan scraping

Dua jalur: **Overpass API** (OpenStreetMap) dengan cache di disk, dan **scraping** tabel kecamatan dari
Wikipedia secara etis. Jalur ketiga, tabel BPS, dibahas di `03_membersihkan_bps`.

In [ ]:
import sys
sys.path.insert(0, "../scripts")

import hashlib
import io
import json
import time

import pandas as pd
import requests

from kota import CFG, RAW

UA = {"User-Agent": "kursus-analitik-perkotaan/1.0"}
CACHE = RAW.parents[1] / "cache" / "http"
CACHE.mkdir(parents=True, exist_ok=True)

## 1. Overpass API, dengan cache
Jalankan ulang notebook tidak boleh memanggil API berulang kali. Respons disimpan ke disk dengan nama
berkas dari *hash* kueri; pemanggilan berikutnya membaca berkas itu.

In [ ]:
SERVER = ["https://overpass-api.de/api/interpreter", "https://maps.mail.ru/osm/tools/overpass/api/interpreter"]


def overpass(query, tries=4):
    key = hashlib.sha1(query.encode()).hexdigest()[:16]
    f = CACHE / f"overpass_{key}.json"
    if f.exists():
        print("cache:", f.name)
        return json.loads(f.read_text())
    for i in range(tries):
        url = SERVER[i % len(SERVER)]
        r = requests.post(url, data={"data": query}, headers=UA, timeout=180)
        if r.status_code in (429, 502, 503, 504):  # server sibuk: tunggu makin lama, lalu coba server lain
            print(f"{r.status_code} dari {url.split('/')[2]}, tunggu {10 * 2 ** i} detik")
            time.sleep(10 * 2 ** i)
            continue
        r.raise_for_status()
        f.write_text(r.text)
        print("unduh:", f.name, len(r.content) // 1024, "KB")
        return r.json()
    raise RuntimeError("Overpass tidak merespons; coba lagi nanti")


w, s, e, n = CFG["bbox"]
q = f"""
[out:json][timeout:120];
nwr["amenity"~"^(clinic|hospital|doctors)$"]({s},{w},{n},{e});
out center tags;
"""
data = overpass(q)
rows = []
for el in data["elements"]:
    c = el.get("center", el)
    t = el.get("tags", {})
    rows.append({"osm_id": f"{el['type'][0]}{el['id']}", "nama": t.get("name"), "amenity": t.get("amenity"),
                 "lat": c.get("lat"), "lon": c.get("lon")})
faskes = pd.DataFrame(rows)
print(len(faskes), "fasilitas kesehatan dalam kotak batas")
faskes["amenity"].value_counts()

Jalankan sel di atas sekali lagi: kali ini data dibaca dari cache, tanpa memanggil server.

In [ ]:
faskes.to_csv(RAW / "faskes_overpass.csv", index=False)
faskes.head()

## 2. Scraping tabel kecamatan dari Wikipedia
Sebelum *scraping*: baca `robots.txt`, batasi kecepatan permintaan, dan kenali diri Anda lewat User-Agent.
Wikipedia mengizinkan akses halaman artikel; untuk data dalam jumlah besar gunakan API atau *dump* mereka.

In [ ]:
robots = requests.get("https://id.wikipedia.org/robots.txt", headers=UA, timeout=30).text
print("Disallow: /wiki/ ?", "Disallow: /wiki/\n" in robots)

url = f"https://id.wikipedia.org/wiki/Kota_{CFG['name'].replace(' ', '_')}"
html = requests.get(url, headers=UA, timeout=30).text
tabel = [t for t in pd.read_html(io.StringIO(html)) if "Kode Kemendagri" in " ".join(map(str, t.columns))]
kec = tabel[0]
kec.columns = [" ".join(c) if isinstance(c, tuple) else c for c in kec.columns]
kec = kec.rename(columns=lambda c: c.split("[")[0].strip())
kec = kec[["Kode Kemendagri", "Kecamatan", "Jumlah Kelurahan"]]
kec = kec[kec["Kode Kemendagri"].astype(str).str.match(r"\d{2}\.\d{2}\.\d{2}")]
kec["Jumlah Kelurahan"] = pd.to_numeric(kec["Jumlah Kelurahan"])
print(len(kec), "kecamatan,", kec["Jumlah Kelurahan"].sum(), "kelurahan")
kec.to_csv(RAW / "kecamatan_wikipedia.csv", index=False)
kec